# Graph load 점검
합성 CSV로 드라이런을 확인하고, 설정된 경우 특정 run의 삭제 드라이런을 확인합니다.


In [ ]:
from pathlib import Path
import csv
import os
import tempfile

CSV_DIR = Path(tempfile.mkdtemp(prefix='graph-load-check-'))
from payment_card.file_pipeline.graph import load


## 합성 CSV 작성과 드라이런


In [ ]:
fixtures = {
    'pf_load_runs.csv': (['run_id', 'input_folder_name', 'model_name', 'started_at', 'node_count', 'edge_count'], [['check-run', 'synthetic', 'demo', '2026-01-01T00:00:00+00:00', 1, 0]]),
    'pf_nodes.csv': (['id', 'node_type', 'props', 'chunk_id', 'run_id'], [['check-node', 'Benefit', '{}', 'synthetic-chunk', 'check-run']]),
    'pf_edges.csv': (['id', 'source_id', 'target_id', 'edge_type', 'run_id'], []),
    'pf_evidence.csv': (['node_id', 'edge_id', 'doc_sha256', 'page_no', 'start_pos', 'end_pos', 'quote_status'], []),
}
for filename, (headers, rows) in fixtures.items():
    with (CSV_DIR / filename).open('w', newline='', encoding='utf-8') as stream:
        writer = csv.writer(stream)
        writer.writerow(headers)
        writer.writerows(rows)
load.load_csvs(CSV_DIR)


## run 삭제 드라이런
가짜 커서가 각 `SELECT COUNT`에 0을 반환하므로 DB에 연결하지 않습니다.


In [ ]:
class FakeDeleteCursor:
    def __enter__(self): return self
    def __exit__(self, *args): return False
    def execute(self, sql, params):
        assert sql.startswith('SELECT COUNT(*) FROM public.pf_')
        assert params in (('synthetic-run-for-check',), ('synthetic-run-for-check', 'synthetic-run-for-check'))
    def fetchone(self): return (0,)

class FakeDeleteConnection:
    def cursor(self): return FakeDeleteCursor()
    def close(self): pass

load._connect = lambda _: FakeDeleteConnection()
load.delete_run('synthetic-run-for-check')
